In [13]:
import os
import numpy as np
import pandas as pd
import pickle
import torch
from sklearn.ensemble import RandomForestClassifier
from google.colab import drive

In [14]:
drive.mount('/content/drive', force_remount=True)

# Path Configuration
DRIVE_PATH = '/content/drive/MyDrive/CyberThreatDetectionSystem_Project/'
DATA_PATH = os.path.join(DRIVE_PATH, 'Data/processed/')
FILEPATH = os.path.join(DATA_PATH, 'IDS-IoT-2024_cleaned.csv')

# Create directories if they don't exist
os.makedirs(DATA_PATH, exist_ok=True)
print(f"Working with data in: {DATA_PATH}")

Mounted at /content/drive
Working with data in: /content/drive/MyDrive/CyberThreatDetectionSystem_Project/Data/processed/


In [15]:
# Data Loading
print("Loading dataset...")
df = pd.read_csv(FILEPATH)
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

# Data quality checks
assert df.duplicated().sum() == 0, "Duplicates found in DataFrame"
assert df.isna().sum().sum() == 0, "NaN values found in DataFrame"
print("No duplicates or NaN values found")

# Label encoding
LABEL_COL = 'Attack_Category_x'
if LABEL_COL in df.columns:
    # Create binary label column
    df['Label'] = np.where(
        df[LABEL_COL].str.upper() == 'NORMAL', 0, 1).astype(np.int64)
    df.drop(columns=[LABEL_COL], inplace=True)
    print("Created binary 'Label' column (0=Normal, 1=Attack)")

    # Compute and cast label distribution to Python ints
    raw_counts = df['Label'].value_counts().to_dict()
    label_counts = {label: int(count) for label, count in raw_counts.items()}
    print(f"Label distribution: {label_counts}")

Loading dataset...
Dataset loaded: 66562 rows, 97 columns
No duplicates or NaN values found
Created binary 'Label' column (0=Normal, 1=Attack)
Label distribution: {1: 57634, 0: 8928}


In [16]:
# Network Flow-Based Splitting
def split_ids_dataset(df, train_size=0.8, val_size=0.1, test_size=0.1, random_seed=42):
    """
    Split IDS dataset by network flows to prevent leakage between train/val/test

    Args:
        df: Pandas DataFrame with network flow data
        train_size/val_size/test_size: Split proportions (should sum to 1)
        random_seed: For reproducibility

    Returns:
        train_df, val_df, test_df: Split dataframes
    """
    print("\nCreating Flow-Based Splits")

    # Identify columns that can define a network flow
    flow_columns = []

    # Port columns (different naming conventions in different datasets)
    port_pairs = [
        ('SPort', 'DPort'),
        ('Source Port', 'Destination Port')
    ]

    for src_port, dst_port in port_pairs:
        if src_port in df.columns and dst_port in df.columns:
            flow_columns.extend([src_port, dst_port])
            break

    # IP address columns (different naming conventions)
    ip_pairs = [
        ('Source IP', 'Destination IP'),
        ('Src IP', 'Dst IP')
    ]

    for src_ip, dst_ip in ip_pairs:
        if src_ip in df.columns and dst_ip in df.columns:
            flow_columns.extend([src_ip, dst_ip])
            break

    # Additional flow identifiers
    potential_flow_ids = [
        'Stream index',  # TCP session identifier
        'TCP Seq No',    # TCP sequence number
        'Protocol',      # Transport protocol
        'Ack No',        # TCP acknowledgment number
        'Window'         # TCP window size
    ]

    for col in potential_flow_ids:
        if col in df.columns:
            flow_columns.append(col)

    if not flow_columns:
        raise ValueError("Couldn't find suitable columns to identify network flows")

    print(f"Creating flow identifiers using: {flow_columns}")

    # Create a flow identifier by hashing the combination of selected columns
    df['flow_id'] = df[flow_columns].astype(str).apply(lambda x: hash(tuple(x)), axis=1)

    # Get unique flows and split them randomly
    unique_flows = df['flow_id'].unique()
    print(f"Found {len(unique_flows)} unique network flows")

    # Shuffle flows for random assignment to splits
    np.random.seed(random_seed)
    np.random.shuffle(unique_flows)

    # Calculate split indices
    n_flows = len(unique_flows)
    train_idx = int(n_flows * train_size)
    val_idx = int(n_flows * (train_size + val_size))

    # Assign flows to splits
    train_flows = unique_flows[:train_idx]
    val_flows = unique_flows[train_idx:val_idx]
    test_flows = unique_flows[val_idx:]

    # Create the split dataframes
    train_df = df[df['flow_id'].isin(train_flows)].copy()
    val_df = df[df['flow_id'].isin(val_flows)].copy()
    test_df = df[df['flow_id'].isin(test_flows)].copy()

    # Clean up temporary column
    for split_df in [train_df, val_df, test_df]:
        split_df.drop(columns=['flow_id'], inplace=True)

    print(f"Split sizes: Train={len(train_df)} rows, Val={len(val_df)} rows, Test={len(test_df)} rows")

    return train_df, val_df, test_df

In [17]:
# Feature Selection
def select_top_features(train_df, val_df, test_df, n_features=20):
    """
    Select top N features using Random Forest importance

    Args:
        train_df/val_df/test_df: Split dataframes
        n_features: Number of top features to select

    Returns:
        train_df_selected, val_df_selected, test_df_selected: Dataframes with selected features
        selected_features: List of selected feature names
    """
    print(f"\nSelecting Top {n_features} Features")

    # Determine columns to exclude from feature selection
    exclude_cols = ['Label']
    time_cols = ['Timestamp', 'Time', 'Time_stamp']

    for col in time_cols:
        if col in train_df.columns:
            exclude_cols.append(col)
            print(f"Detected timestamp column: {col}")

    # Prepare training data for feature selection
    X_train = train_df.drop(columns=exclude_cols)
    y_train = train_df['Label']

    print(f"Training Random Forest on {X_train.shape[1]} features")

    # Initialize and fit Random Forest for feature importance
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,  # Limit depth for faster training
        random_state=42,
        n_jobs=-1      # Use all available CPUs
    )
    rf.fit(X_train, y_train)

    # Get feature importances
    importances = rf.feature_importances_
    feature_names = X_train.columns

    # Create DataFrame of features and their importances
    feature_importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=False)

    # Print top features
    print("\nTop features by importance:")
    for i, row in feature_importance_df.head(n_features).iterrows():
        print(f"{row['Feature']}: {row['Importance']:.4f}")

    # Select the top N features
    top_features = feature_importance_df['Feature'].head(n_features).tolist()

    # Add back excluded columns
    for col in exclude_cols:
        if col in train_df.columns:
            top_features.append(col)

    # Create new dataframes with only selected features
    train_df_selected = train_df[top_features].copy()
    val_df_selected = val_df[top_features].copy()
    test_df_selected = test_df[top_features].copy()

    print(f"Selected {len(top_features)} features (including Label and timestamp if present)")

    return train_df_selected, val_df_selected, test_df_selected, top_features


In [18]:
# Leakage Verification
def verify_no_leakage(train_df, val_df, test_df):
    """
    Verify there's no data leakage between splits

    Args:
        train_df/val_df/test_df: Split dataframes

    Returns:
        bool: True if no leakage detected
    """

    # Exclude non-feature columns
    exclude_cols = ['Label']
    time_cols = ['Timestamp', 'Time', 'Time_stamp']

    for col in time_cols:
        if col in train_df.columns:
            exclude_cols.append(col)

    feature_cols = [col for col in train_df.columns if col not in exclude_cols]

    # Create fingerprints of each record (tuple of feature values)
    print(f"Creating fingerprints using {len(feature_cols)} features")

    # Convert to sets for faster intersection operations
    train_fingerprints = set(map(tuple, train_df[feature_cols].values))
    val_fingerprints = set(map(tuple, val_df[feature_cols].values))
    test_fingerprints = set(map(tuple, test_df[feature_cols].values))

    # Check for overlaps between splits
    train_val_overlap = len(train_fingerprints.intersection(val_fingerprints))
    train_test_overlap = len(train_fingerprints.intersection(test_fingerprints))
    val_test_overlap = len(val_fingerprints.intersection(test_fingerprints))

    total_overlap = train_val_overlap + train_test_overlap + val_test_overlap

    # Report findings
    if total_overlap > 0:
        print(f" WARNING: Found {total_overlap} overlapping records between splits:")
        print(f"  - Train-Val overlap: {train_val_overlap}")
        print(f"  - Train-Test overlap: {train_test_overlap}")
        print(f"  - Val-Test overlap: {val_test_overlap}")
        print("Your evaluation metrics may be artificially inflated!")
    else:
        print("No data leakage detected between splits")

    return total_overlap == 0


In [19]:
# Graph Creation
def create_graph_data(train_df, val_df, test_df, selected_features, save_path, time_window=5):
    """
    Create graph representations from the split dataframes for GNN training

    Args:
        train_df/val_df/test_df: Split dataframes
        selected_features: List of feature names
        save_path: Path to save graph data
        time_window: Number of previous nodes to connect to

    Returns:
        train_graph, val_graph, test_graph: Graph data dictionaries
    """

    # Helper function to convert dataframe to graph
    def df_to_graph(df, set_name):
        print(f"Creating {set_name} graph...")

        # Extract features and labels
        features = df.drop(columns=['Label']).values
        labels = df['Label'].values

        # Create a graph where each node is a network packet
        n_nodes = len(df)
        edge_list = []

        # Connect each node to k previous nodes (simulating temporal connections)
        for i in range(n_nodes):
            # Connect to 'time_window' previous nodes (temporal connection)
            start = max(0, i-time_window)
            for j in range(start, i):
                edge_list.append([j, i])  # Directed edge from past to present

        print(f"  - Created {len(edge_list)} edges for {n_nodes} nodes")

        # Convert to PyTorch Geometric format
        if len(edge_list) > 0:
            edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
        else:
            # Empty graph fallback
            edge_index = torch.zeros((2, 0), dtype=torch.long)

        # Return as dictionary for easy serialization
        return {
            'x': features.astype(np.float32),
            'edge_index': edge_index.numpy(),
            'y': labels.astype(np.int64),
            'timestamps': np.arange(n_nodes).astype(np.float32)  # Artificial timestamps
        }

    # Create graphs for each split
    train_graph = df_to_graph(train_df, "Training")
    val_graph = df_to_graph(val_df, "Validation")
    test_graph = df_to_graph(test_df, "Testing")

    # Save graphs to disk
    os.makedirs(save_path, exist_ok=True)

    with open(os.path.join(save_path, 'train_graph.pkl'), 'wb') as f:
        pickle.dump(train_graph, f)

    with open(os.path.join(save_path, 'val_graph.pkl'), 'wb') as f:
        pickle.dump(val_graph, f)

    with open(os.path.join(save_path, 'test_graph.pkl'), 'wb') as f:
        pickle.dump(test_graph, f)

    # Save feature names for reference
    feature_cols = [col for col in train_df.columns if col != 'Label']
    with open(os.path.join(save_path, 'feature_names.pkl'), 'wb') as f:
        pickle.dump(feature_cols, f)

    print(f"\nGraph data saved to {save_path}")
    print(f"Train graph: {len(train_graph['x'])} nodes, {train_graph['edge_index'].shape[1]} edges")
    print(f"Val graph: {len(val_graph['x'])} nodes, {val_graph['edge_index'].shape[1]} edges")
    print(f"Test graph: {len(test_graph['x'])} nodes, {test_graph['edge_index'].shape[1]} edges")

    # Check class distribution in graphs
    train_pos = (train_graph['y'] == 1).sum()
    val_pos = (val_graph['y'] == 1).sum()
    test_pos = (test_graph['y'] == 1).sum()

    print("\nClass distribution in graphs:")
    print(f"Train: {train_pos}/{len(train_graph['y'])} positive ({train_pos/len(train_graph['y']):.2%})")
    print(f"Val: {val_pos}/{len(val_graph['y'])} positive ({val_pos/len(val_graph['y']):.2%})")
    print(f"Test: {test_pos}/{len(test_graph['y'])} positive ({test_pos/len(test_graph['y']):.2%})")

    return train_graph, val_graph, test_graph

In [20]:
# Complete Pipeline
def prepare_ids_data_for_gnn(df, top_n_features=20, save_path=DATA_PATH):
    """
    Complete pipeline to prepare IDS data for GNN model

    Args:
        df: Input DataFrame with IDS data
        top_n_features: Number of top features to select
        save_path: Path to save processed data

    Returns:
        train_graph, val_graph, test_graph: Graph data for GNN model
    """

    # Step 1: Split the dataset by network flows
    train_df, val_df, test_df = split_ids_dataset(df)

    # Step 2: Select top features
    train_df_selected, val_df_selected, test_df_selected, selected_features = select_top_features(
        train_df, val_df, test_df, n_features=top_n_features
    )

    # Step 3: Verify no leakage
    no_leakage = verify_no_leakage(train_df_selected, val_df_selected, test_df_selected)

    # Step 4: Save the processed dataframes
    save_splits(train_df_selected, val_df_selected, test_df_selected, save_path)

    # Step 5: Create graph representations
    train_graph, val_graph, test_graph = create_graph_data(
        train_df_selected, val_df_selected, test_df_selected,
        selected_features, save_path
    )

    return train_graph, val_graph, test_graph


In [21]:
# Save Splits
def save_splits(train_df, val_df, test_df, save_path):
    """
    Save the train/val/test splits to CSV files

    Args:
        train_df/val_df/test_df: Split dataframes
        save_path: Path to save CSV files
    """
    print("\n Saving DataFrame Splits")

    os.makedirs(save_path, exist_ok=True)
    train_df.to_csv(os.path.join(save_path, 'train_split.csv'), index=False)
    val_df.to_csv(os.path.join(save_path, 'val_split.csv'), index=False)
    test_df.to_csv(os.path.join(save_path, 'test_split.csv'), index=False)

    print(f"Splits saved to {save_path}")

In [22]:
# Run Pipeline
# Main execution block
print("\nStarting Main Execution")

# Split the dataset by network flows
train_df, val_df, test_df = split_ids_dataset(df)

# Select top features
train_df_selected, val_df_selected, test_df_selected, selected_features = select_top_features(
    train_df, val_df, test_df, n_features=20
)

# Verify no leakage
verify_no_leakage(train_df_selected, val_df_selected, test_df_selected)

# Save the processed dataframes
save_splits(train_df_selected, val_df_selected, test_df_selected, DATA_PATH)

# Create graph representations
train_graph, val_graph, test_graph = create_graph_data(
    train_df_selected, val_df_selected, test_df_selected,
    selected_features, DATA_PATH
)

print("\nGraph Creation Complete!")


Starting Main Execution

Creating Flow-Based Splits
Creating flow identifiers using: ['SPort', 'DPort', 'Stream index', 'TCP Seq No', 'Ack No', 'Window']
Found 60147 unique network flows
Split sizes: Train=53447 rows, Val=6888 rows, Test=6227 rows

Selecting Top 20 Features
Training Random Forest on 96 features

Top features by importance:
Length: 0.2066
Length.1: 0.1465
DPort: 0.0745
Ack No: 0.0595
TCP Seq No: 0.0493
SPort: 0.0457
Type: 0.0353
Source Port: 0.0349
Stream index: 0.0315
Response time: 0.0271
Destination Port: 0.0267
Acknowledgment_Not set: 0.0240
TCP Segment Length: 0.0220
Duplicate ACK #: 0.0187
Message type: 0.0157
Kind_0: 0.0134
Acknowledgment_0: 0.0132
Push_Not set: 0.0123
Timestamp value: 0.0119
Acknowledgment_Set: 0.0114
Selected 21 features (including Label and timestamp if present)
Creating fingerprints using 20 features
No data leakage detected between splits

 Saving DataFrame Splits
Splits saved to /content/drive/MyDrive/CyberThreatDetectionSystem_Project/Dat